# 06.01 — FastAPI Integration

Orthograph produces **typed query contracts** via `ReadQuery` and `WriteQuery`.
Orthograph is a **typed query-contract and validation layer — not an ORM.** It does not
generate CRUD, does not manage sessions, and does not map objects to rows. It gives you
validated Cypher, typed params, and a typed result shape; you wire those to your web
framework. This notebook demonstrates that wiring against a real FastAPI service and
verifies the HTTP interface end-to-end using `TestClient` — no external server required.

**Key constraint (ADR-024):** `src/orthograph/` must not acquire a hard or optional
dependency on FastAPI. The wiring patterns shown here live in the consuming application
layer. Orthograph is the contract layer only.

Sections:
1. Dependency check — skip gracefully if FastAPI is not installed
2. Domain model and queries — `Movie`, `Director`, `DIRECTED`, `MoviesByYear`, `CreateMovie`, `MoviesByDirector`, `CreateDirected`
2b. Graph definition + semantic validation — `validate_cypher` against `GraphDefinition`
3. Schema leakage proof — `NodeModel` as a safe HTTP response model
4. `materialize()` — the mapping seam, and when the 1:1 shortcut is valid
5. In-memory store and fake executor — replaces a live graph database
6. FastAPI application — routes wired from the query `Output` classes
7. POST — create a movie (writes return mutation counters, never the created object)
8. GET — query movies and verify the typed response
9. Projection (DTO) escape hatch — only for shapes that do NOT map 1:1 to a node
10. OpenAPI schema inspection
11. Limitations and future work

## 1. Dependency check

This notebook requires `fastapi` and `httpx`. They are **not** dependencies of
`orthograph` itself. The cell below skips the rest of the notebook with a clear
message if either package is missing.

In [ ]:
import importlib
from typing import Any, Optional

import fastapi
import httpx
from pydantic import BaseModel

from orthograph.cypher.base_models import CypherReadQuery, CypherWriteQuery
from orthograph.cypher.query_execution import CypherExecutor
from orthograph.graph_definition.graph_definition import GraphDefinition


_MISSING = [
    pkg for pkg in ("fastapi", "httpx") if importlib.util.find_spec(pkg) is None
]


if _MISSING:
    msg = (
        f"Missing packages: {', '.join(_MISSING)}\n"
        "Install them with:\n"
        f"    pip install {' '.join(_MISSING)}\n"
        "This notebook is skipped — orthograph itself does not require FastAPI."
    )

    print(msg)

    # Raise to halt kernel execution of remaining cells cleanly.

    raise SystemExit(msg)


print(f"fastapi {fastapi.__version__}  |  httpx {httpx.__version__}  — OK")

## 2. Domain model and queries

A minimal filmography domain: one `NodeModel`, one `ReadQuery`, and one `WriteQuery`
that declares an `Output`.

`CypherReadQuery[Params, Output]` auto-populates `Query.Params` and `Query.Output`
from the generic arguments — no repetition required.

### Cypher parameters are written `$name` — never `` `$name` ``

Value parameters use a bare `$released`. Do **not** wrap them in backticks: in Cypher a
backtick-quoted token is an *escaped identifier* (a label/property name), not a parameter,
so the driver would never bind it and the query would silently fail to filter. The
declarative base validates the template at class-definition time (`CypherReadQuery`
parses it and checks `$param` ↔ `Params` 1:1 alignment).

### Definition-time validation (no full GraphDefinition needed)

This demo does **not** build a full `GraphDefinition`, so we get *syntactic* validation
(does the Cypher parse? do the `$param` placeholders line up 1:1 with the `Params`
fields?) but **not** semantic validation against a schema (are `Movie` and `released`
real model entities?). Semantic validation is `validate_cypher(query, graph_definition)`
and is out of scope here.

> **Known gap (tech-debt E20/T7):** the current parser accepts a backtick-wrapped
> parameter `` `$released` `` as if it were valid, because graphglot lexes it as a single
> escaped-identifier token. Syntactic validation is therefore *necessary but not
> sufficient* — it will not catch that specific mistake. The templates below use the
> correct bare-`$param` form.

In [ ]:
import warnings

from orthograph.cypher.parser import parse_cypher
from orthograph.graph_definition.models import NodeModel, RelationshipModel
from orthograph.query.base_models import QueryBackedReadPort, ReadPort

In [ ]:
# --- Domain model ---


class Movie(NodeModel):
    __label__ = "Movie"
    __uid_field__ = "title"
    title: str
    released: int
    tagline: Optional[str] = None


# --- Read query ---


class MoviesByYearParams(BaseModel):
    released: int


class MoviesByYear(CypherReadQuery[MoviesByYearParams, Movie]):
    """Return all movies released in a given year."""

    name = "movies_by_year"
    # Parameters are bare $name — NOT backtick-wrapped. See the §2 note.
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released, m.tagline AS tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        # RETURN aliases (title/released/tagline) map 1:1 to Movie fields, so the
        # 1:1 shortcut is valid here. See §4 for when it is NOT.
        return Movie.model_validate(raw)


# --- Write result model ---
# This is a plain BaseModel — not a NodeModel — because the write result
# (mutation counters) has no graph-node identity.


class CreateMovieResult(BaseModel):
    """Structured result returned by CreateMovie to the API caller."""

    nodes_created: int


# --- Write query with Output ---


class CreateMovieParams(BaseModel):
    title: str
    released: int


class CreateMovie(CypherWriteQuery[CreateMovieParams, CreateMovieResult]):
    """Create a Movie node and return a structured result.

    Note: CypherExecutor.write() discards any RETURN rows from the Cypher
    template and passes a CypherWriteResultSummary (mutation counters only)
    to interpret_result. A write CANNOT return the created node — see §7/§11.
    """

    name = "create_movie"
    Output = CreateMovieResult
    # Parameters are bare $name — NOT backtick-wrapped. See the §2 note.
    cypher_template = "CREATE (m:Movie {title: $title, released: $released})"

    def interpret_result(self, raw: Any) -> CreateMovieResult:
        # raw is a CypherWriteResultSummary (satisfies WriteResultSummary protocol)
        return CreateMovieResult(nodes_created=raw.nodes_created)


# --- Additional models ---


class Director(NodeModel):
    __label__ = "Director"
    __uid_field__ = "name"
    name: str
    born: Optional[int] = None


class Directed(RelationshipModel):
    __label__ = "DIRECTED"
    __source_label__ = "Director"
    __target_label__ = "Movie"
    # No properties in this domain.


# --- Read query: movies by director (traverses the relationship) ---


class DirectedParams(BaseModel):
    director_name: str


class MoviesByDirector(CypherReadQuery[DirectedParams, Movie]):
    """Return all movies directed by a given director."""

    name = "movies_by_director"
    cypher_template = (
        "MATCH (d:Director {name: $director_name})-[:DIRECTED]->(m:Movie) "
        "RETURN m.title AS title, m.released AS released, m.tagline AS tagline"
    )

    def materialize(self, raw: dict[str, Any]) -> Movie:
        return Movie.model_validate(raw)  # 1:1 columns -> fields (see §4)


# --- Write query: create a DIRECTED relationship ---


class CreateDirectedParams(BaseModel):
    director_name: str
    movie_title: str


class CreateDirectedResult(BaseModel):
    """Mutation counters from linking a director to a movie."""

    relationships_created: int


class CreateDirected(CypherWriteQuery[CreateDirectedParams, CreateDirectedResult]):
    """Create a DIRECTED relationship between an existing Director and Movie.

    write() discards RETURN rows — only mutation counters are available (see §7/§10).
    """

    name = "create_directed"
    Output = CreateDirectedResult
    cypher_template = (
        "MATCH (d:Director {name: $director_name}), (m:Movie {title: $movie_title}) "
        "CREATE (d)-[:DIRECTED]->(m)"
    )

    def interpret_result(self, raw: Any) -> CreateDirectedResult:
        return CreateDirectedResult(relationships_created=raw.relationships_created)


# Definition-time validation ran at class-definition time (CypherReadQuery/WriteQuery
# parse the template and check $param alignment). Re-run explicitly here to make the
# syntactic-validation stage visible. SYNTAX only — semantic check is in §2b.
for q in (MoviesByYear(), CreateMovie(), MoviesByDirector(), CreateDirected()):
    parse_cypher(q.cypher_template)
    print(f"cypher parses OK: {q.name}")

print()
print("Movie         fields  :", list(Movie.model_fields.keys()))
print("Director      fields  :", list(Director.model_fields.keys()))
print("MoviesByYear  Output  :", MoviesByYear.Output.__name__)
print("CreateMovie   Output  :", CreateMovie.Output.__name__)
print("MoviesByDirector      :", MoviesByDirector.Output.__name__)
print("CreateDirected Output :", CreateDirected.Output.__name__)

## 2b. Graph definition and semantic validation

Syntactic validation (§2) tells you the Cypher parses and the `$param` ↔ `Params`
alignment is 1:1. It does **not** tell you whether the labels, relationship types,
or properties you reference actually exist in your declared schema.

**Semantic validation** requires a `GraphDefinition`. We build one here from the two
`NodeModel` subclasses and the `RelationshipModel`, then call `validate_cypher` against
each query template. This is the second tier of the two-tier check:

| Tier | What it checks | Tool |
|------|---------------|------|
| Syntactic (§2) | Cypher parses; `$param` ↔ `Params` 1:1 | `parse_cypher(template)` |
| Semantic (§2b) | Labels, rel-types, properties match the declared schema | `validate_cypher(cypher, graph_definition)` |

> Semantic validation catches typos like `Moovie` (wrong label) or `relased`
> (wrong property name) that syntactic parse alone misses.


In [ ]:
from orthograph.cypher.parser import validate_cypher


# --- Build the GraphDefinition from the declared node and relationship models ---
# GraphDefinition takes lists at construction — no add_* methods.

gd = GraphDefinition(
    "filmography",
    node_types=[Movie, Director],
    relationship_types=[Directed],
)

print("Registered node labels        :", sorted(gd.node_labels))
print("Registered relationship labels:", sorted(gd.relationship_labels))
print()

# --- Semantic validation of query templates ---

for query_cls_name, template in [
    ("MoviesByYear", MoviesByYear.cypher_template),
    ("MoviesByDirector", MoviesByDirector.cypher_template),
    ("CreateDirected", CreateDirected.cypher_template),
]:
    result = validate_cypher(template, gd)
    status = "VALID" if result.is_valid else "INVALID"
    print(f"{status:8s}  {query_cls_name}")
    for issue in result.issues:
        print(f"          [{issue.severity.value}] {issue.code}: {issue.message}")

# --- Demonstrate that a typo IS caught at the semantic tier but not by parse_cypher ---

print()
bad_template = "MATCH (m:Moovie) RETURN m.title AS title"
try:
    parse_cypher(bad_template)
    print("Syntax check: PASSES (graphglot does not know your schema)")
except Exception as e:
    print("Syntax check: FAILED:", e)

bad_result = validate_cypher(bad_template, gd)
print(
    "Schema check:",
    "VALID" if bad_result.is_valid else "INVALID — schema catches the typo",
)
for issue in bad_result.issues:
    print(f"  [{issue.severity.value}] {issue.code}: {issue.message}")

## 3. Schema leakage proof — `NodeModel` as a safe HTTP response model

A key design question: does `Movie` (a `NodeModel`) leak graph-specific internals
(`__label__`, `__uid_field__`, Neo4j internal ids) into the HTTP/JSON response?

The answer is **no**. All graph metadata is declared as `ClassVar` — Pydantic
excludes `ClassVar` attributes from instance fields, `model_dump()`, and
`model_json_schema()` entirely. The serialized shape contains **only** the typed
domain fields the author declared.

### When you can skip the DTO — and the exact boundary

Because graph internals do not leak, `Movie` is safe to use directly as
`response_model=` **in one specific case**: when the query's `RETURN` columns map
**1:1** to the node's fields and you want to expose the full node shape unchanged.
In that case `NodeModel` is the graph definition **and** the response model with no
extra glue.

This is a narrow exemption, not a general "orthograph replaces your DTOs" claim.
Orthograph is a validator, not an ORM: the moment the API shape diverges from the
stored shape by even one field, you are back to a DTO (a projection `BaseModel`) and
an explicit mapping in `materialize()`.

### When you still need a DTO

A separate response model is necessary whenever the API contract diverges from the
stored node shape: renaming a field, hiding sensitive properties, adding computed
fields, combining data from multiple nodes, or versioning the API independently of
the storage schema. The library does not eliminate that need — it makes the DTO
**optional only for the strict 1:1 case**. §4 shows the mapping seam; §9 shows a
concrete projection (`MovieSummary`) for when a DTO is the right choice.

In [ ]:
schema = Movie.model_json_schema()
schema_props = set(schema["properties"].keys())

print("Movie JSON schema properties:", sorted(schema_props))

# Only the three declared domain fields — no __label__, no __uid_field__,
# no internal id, no relationship fields.
assert schema_props == {"title", "released", "tagline"}, (
    f"Unexpected properties leaked into schema: {schema_props}"
)

# ClassVar dunders are NOT in model_fields.
instance_fields = set(Movie.model_fields.keys())
assert "__label__" not in instance_fields
assert "__uid_field__" not in instance_fields

# model_dump() on a Movie instance contains only domain data.
m = Movie(title="The Matrix", released=1999)
dumped = m.model_dump()
assert set(dumped.keys()) == {"title", "released", "tagline"}
print("model_dump()           :", dumped)
print("NodeModel is DTO-free — no leakage.")

## 4. `materialize()` — the mapping seam

`materialize()` maps **one raw graph record** (a `dict` of the `RETURN` columns) to the
declared `Output` type. It is the single place where storage shape meets API shape.
There are two cases, and only one of them has a shortcut.

**1:1 case — columns match fields.** When every `RETURN` alias equals an `Output`
field name and the types line up, `materialize` is mechanical and you can use
`Output.model_validate(raw)` (or `Output(**raw)`) instead of hand-listing fields. This
is the *only* situation in which the per-field boilerplate disappears.

**Divergent case — columns differ from fields.** The moment a column is renamed,
dropped, computed, or sourced from multiple nodes, the 1:1 shortcut is invalid and you
**must** write the mapping explicitly. `materialize` is exactly the DTO seam — see §9.

The cell below proves both: the 1:1 shortcut produces the same `Movie` as the explicit
constructor, and a renamed column requires the explicit form.

In [ ]:
# A raw record as it would arrive from the driver for the MoviesByYear RETURN clause.
raw_record = {"title": "The Matrix", "released": 1999, "tagline": None}

# 1:1 shortcut — valid because RETURN aliases == Movie field names.
via_shortcut = Movie.model_validate(raw_record)

# Explicit form — always valid, required when columns diverge from fields.
via_explicit = Movie(
    title=raw_record["title"],
    released=raw_record["released"],
    tagline=raw_record.get("tagline"),
)

assert via_shortcut == via_explicit
print("1:1 shortcut == explicit :", via_shortcut == via_explicit)

# Divergent case: a record whose column ('release_year') does NOT match the field
# ('released'). The shortcut fails; an explicit mapping is mandatory.
divergent = {"title": "Heat", "release_year": 1995}
try:
    Movie.model_validate(divergent)  # missing 'released' -> ValidationError
    raise AssertionError("expected the 1:1 shortcut to fail on a renamed column")
except Exception as exc:
    print("shortcut fails on renamed column :", type(exc).__name__)

# The mapping must be written by hand — this is the DTO seam (full demo in §9).
mapped = Movie(title=divergent["title"], released=divergent["release_year"])
print("explicit mapping recovers it     :", mapped.model_dump())

## 5. In-memory store and fake executor

A real executor needs a graph driver. Here we substitute an in-memory list that
acts as the database. The fake session supports both `session.run()` (reads) and
`session.begin_transaction()` (writes) so `CypherExecutor` can be used unchanged.

> **What this fake does and does NOT prove.** The fake session ignores the Cypher
> string and filters its Python list directly, so this notebook verifies the
> **wiring** (params in, typed objects out, HTTP shape) — it does **not** execute or
> verify the `cypher_template` against a real engine. Syntactic validation (§2) is the
> only check the templates actually receive here; a live-driver notebook is deferred
> (§11).

In [ ]:
# --- In-memory store (replaces a live graph database) ---

_MOVIE_STORE: list[dict[str, Any]] = []


class _FakeWriteSummary:
    """Satisfies WriteResultSummary for in-memory writes."""

    def __init__(self) -> None:
        self.nodes_created = 1
        self.nodes_deleted = 0
        self.relationships_created = 0
        self.relationships_deleted = 0
        self.properties_set = 0

    def consume(self) -> "_FakeWriteSummary":
        """CypherWriteResultSummary.from_neo4j_result calls .consume().counters."""
        return self

    @property
    def counters(self) -> "_FakeWriteSummary":
        return self


class _FakeTransaction:
    """Context for a write; appends to the shared store on run()."""

    def __init__(self, store: list[dict[str, Any]]) -> None:
        self._store = store
        self._summary = _FakeWriteSummary()

    def run(self, cypher: str, **params: Any) -> "_FakeWriteSummary":
        if "CREATE" in cypher:
            self._store.append(
                {
                    "title": params.get("title"),
                    "released": params.get("released"),
                    "tagline": params.get("tagline"),
                }
            )
        return self._summary

    def commit(self) -> None:
        pass

    def rollback(self) -> None:
        pass


class _FakeSession:
    """Minimal graph session backed by _MOVIE_STORE."""

    def __init__(self, store: list[dict[str, Any]]) -> None:
        self._store = store

    def __enter__(self) -> "_FakeSession":
        return self

    def __exit__(self, *_: Any) -> None:
        pass

    def run(self, cypher: str, **params: Any) -> list[dict[str, Any]]:
        """Filter the store by released year (matches the read query's WHERE clause)."""
        released = params.get("released")
        return [
            row
            for row in self._store
            if released is None or row["released"] == released
        ]

    def begin_transaction(self) -> _FakeTransaction:
        return _FakeTransaction(self._store)


def _make_executor() -> CypherExecutor:
    return CypherExecutor(lambda: _FakeSession(_MOVIE_STORE))


print("Fake executor ready. Store is empty:", _MOVIE_STORE)

## 6. FastAPI application

### Idiom: use the `Output` class directly as `response_model`

Because `Movie` and `CreateMovieResult` are plain Pydantic models, they plug
directly into `response_model=`. The query already declares its `Output` (the second
generic argument), so the same class is the FastAPI response model — no schema
duplication. Each route names its `Output` class directly.

### Read vs write symmetry

**All** read routes are injected via `ReadPort` + `Depends()` — the route never
imports the executor directly. This pattern is used consistently in §6, §9, and §9.
Swap the executor at one place (`get_movies_port`, `get_paged_movies_port`,
`get_summaries_port`) to point at a real driver; all routes follow automatically.

Writes call the executor directly because `WritePort` / `QueryBackedWritePort`
is not yet part of the v0.1 API (see §11 Limitations). The same composition-root
swap still applies: replace `_executor` at one place to point at a real driver.

In [ ]:
from fastapi import Depends, FastAPI
from fastapi.testclient import TestClient


# Suppress the starlette httpx deprecation warning (cosmetic only — TestClient works).
warnings.filterwarnings("ignore", category=DeprecationWarning)

# The query's declared Output IS the FastAPI response_model — no duplication, no
# indirection. (A QueryCatalogue can introspect these via describe()/output_class,
# but the catalogue is an introspection registry, not route-wiring infrastructure:
# it does not compare the contract to the route or catch drift. So we wire the
# Output classes directly and keep the demo honest.)
_MoviesOutput = MoviesByYear.Output  # Movie
_CreateMovieOutput = CreateMovie.Output  # CreateMovieResult
print("read  Output :", _MoviesOutput.__name__)
print("write Output :", _CreateMovieOutput.__name__)
# --- DI factories ---
_executor = _make_executor()


def get_movies_port() -> ReadPort[MoviesByYearParams, Movie]:
    """Dependency factory — swap _executor here to point at a real driver."""
    return QueryBackedReadPort(MoviesByYear(), _executor)


# --- FastAPI app ---
app = FastAPI(title="Filmography API")


# POST: response_model is CreateMovie.Output (CreateMovieResult).
# Writes call the executor directly — WritePort is deferred (see §11).
@app.post("/movies", response_model=_CreateMovieOutput)
def create_movie(body: CreateMovieParams) -> CreateMovieResult:
    """Create a Movie node. Returns a structured write summary (counters only)."""
    return _executor.write(CreateMovie(), body.model_dump())


# GET: response_model is MoviesByYear.Output (Movie) — no DTO, no duplication.
# ReadPort is injected via Depends(); swap the factory to change the executor.
@app.get("/movies", response_model=list[_MoviesOutput])
def get_movies(
    released: int,
    port: ReadPort[MoviesByYearParams, Movie] = Depends(get_movies_port),
) -> list[Movie]:
    """Return all movies released in a given year."""
    return port.fetch(MoviesByYearParams(released=released))


print("\nRegistered routes:")
for route in app.routes:
    if hasattr(route, "methods"):
        print(f"  {sorted(route.methods)} {route.path}")

## 7. POST — create a movie

`TestClient` runs the ASGI app in-process (no network, no server process). The POST
body is serialised as JSON; the response must match `CreateMovieResult`.

### A write returns mutation counters, never the created object

`nodes_created` comes from the executor's `CypherWriteResultSummary` mutation
counters. There is **no** `title` in the response. This is not a demo simplification —
it is a hard property of `CypherExecutor.write()`: it consumes the driver result into
the five mutation counters (`nodes_created`, `nodes_deleted`, `relationships_created`,
`relationships_deleted`, `properties_set`) and **discards any `RETURN` rows** from the
template. `interpret_result` only ever sees counters.

This cannot be generalised away: a `WriteQuery` cannot return the created entity's id
or shape. A REST `POST` that must echo the created resource needs a **follow-up read
query** keyed on the identifier the caller supplied (here, `title`). This is the most
significant ergonomic gap for typical REST APIs — see §11.

In [ ]:
client = TestClient(app)

# Clear the store so this cell is idempotent on re-runs.
_MOVIE_STORE.clear()

response = client.post("/movies", json={"title": "The Matrix", "released": 1999})

print("Status :", response.status_code)
print("Body   :", response.json())

assert response.status_code == 200
body = response.json()
assert body["nodes_created"] == 1
# No 'title' in the response — write results carry counters only.
assert "title" not in body

# Validate the response parses cleanly into the declared Output model.
result = CreateMovieResult.model_validate(body)
print("Parsed :", result)

# Seed two more movies for the GET demonstration.
client.post("/movies", json={"title": "Fight Club", "released": 1999})
client.post("/movies", json={"title": "Speed", "released": 1994})
print("\nStore after inserts:", _MOVIE_STORE)

## 8. GET — query movies by year

The GET route returns `list[Movie]`. FastAPI serialises each item through the
`response_model` (`Movie`) so we can validate the shape directly.

In [ ]:
response = client.get("/movies", params={"released": 1999})

print("Status :", response.status_code)
print("Body   :", response.json())

assert response.status_code == 200
movies = response.json()
assert len(movies) == 2, f"expected 2 movies, got {len(movies)}"

titles = {m["title"] for m in movies}
assert titles == {"The Matrix", "Fight Club"}

# All records must parse cleanly into Movie.
parsed = [Movie.model_validate(m) for m in movies]
print("\nParsed Movie objects:")
for m in parsed:
    print(f"  {m.title} ({m.released})")

# 1994 movies are filtered out.
response_94 = client.get("/movies", params={"released": 1994})
assert len(response_94.json()) == 1
assert response_94.json()[0]["title"] == "Speed"
print("\n1994 results:", response_94.json())

## 9. Projection (DTO) escape hatch — for shapes that do NOT map 1:1 to a node

Using `NodeModel` directly as `response_model` (as in §6 and §9) is correct **only**
when the `RETURN` columns map 1:1 to the node fields. It is not always the case.

A separate DTO (`BaseModel` projection as `Output`) is the correct tool when:

- a field must be renamed in the API without renaming it in the graph,
- a stored field must be hidden from API consumers,
- a computed or joined field needs to appear that is not a graph property,
- results from multiple nodes must be combined into one response shape,
- the API must be versioned independently of the storage schema.

Declare the DTO as a plain `BaseModel` and use it as the `Output` generic argument.
The query's `materialize()` method is the mapping seam: it receives raw graph records
and returns the DTO (the explicit form from §4, because the columns no longer match
1:1). FastAPI is given only the DTO — the `NodeModel` is never exposed.

The library does not eliminate DTOs. It makes them **opt-in**, and only for the strict
1:1 case: skip one when the node shape is the right contract; add one the moment it is
not. Both paths use the same `ReadPort` + `Depends()` wiring — the route does not
change shape.

### Is using `NodeModel` as `Output` an antipattern?

Potentially — but only if you treat it as the default rather than as a deliberate choice.

The type bound on the `Output` generic argument is `D: bound=BaseModel` — any Pydantic
model. `NodeModel` happens to be a `BaseModel` subclass, but it also carries graph-definition
baggage: `__label__`, `__uid_field__`, cardinality specs, `get_property_specs()`. A
consumer that receives a `Movie` can see and depend on `Movie.__label__`. That is a
leaky abstraction — the storage schema bleeds into the API layer.

The correct mental model for the layering is:

```
GraphDefinition (NodeModel)  ← graph contract; used only by the validator
        │
        ▼  validate_cypher() checks labels/properties here — no other coupling
CypherReadQuery[Params, DTO]  ← query result; Output is a plain BaseModel
        │
        ▼  materialize() is the explicit translation seam
Consumer / FastAPI route      ← sees only the DTO; never imports NodeModel
```

The `MoviesByYear` query in §6 uses `Movie` as `Output` because the `RETURN` columns map
1:1 to `Movie` fields and the goal is to demonstrate the full node shape. That is the narrow
exemption. The sub-section below replaces it with a fully-decoupled `MovieCard` DTO — same
Cypher, same route wiring, zero graph internals exposed to the consumer.

In [ ]:
# -----------------------------------------------------------------------
# Replacing the NodeModel with a fully-decoupled DTO
# -----------------------------------------------------------------------
# Same Cypher template, same route, different Output contract.
# The consumer never imports Movie or any NodeModel.


class MovieCard(BaseModel):
    """Decoupled API DTO — no graph concepts, no NodeModel inheritance.

    The graph can rename 'released' to 'release_year' without touching
    this class. Only materialize() needs to be updated in that case.
    The API contract (field names visible to callers) is independent.
    """

    title: str
    year: int  # renamed from 'released' in the graph — explicit seam


class MovieCardsByYear(CypherReadQuery[MoviesByYearParams, MovieCard]):
    """Same query as MoviesByYear — but Output is a plain DTO, not a NodeModel."""

    name = "movie_cards_by_year"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released"
    )

    def materialize(self, raw: dict[str, Any]) -> MovieCard:
        # Explicit translation — graph column 'released' -> DTO field 'year'.
        # This seam absorbs any rename on either side independently.
        return MovieCard(title=raw["title"], year=raw["released"])


# Wire the new query into a ReadPort with the same Depends() idiom.
# The route does not change shape — only the generic parameters differ.
def get_movie_cards_port() -> ReadPort[MoviesByYearParams, MovieCard]:
    """Composition-root factory. Swap _executor here for a real driver."""
    return QueryBackedReadPort(MovieCardsByYear(), _executor)


@app.get("/movie-cards", response_model=list[MovieCard])
def get_movie_cards(
    released: int,
    port: ReadPort[MoviesByYearParams, MovieCard] = Depends(get_movie_cards_port),
) -> list[MovieCard]:
    """Return movie cards (decoupled DTO — no NodeModel in the response contract)."""
    return port.fetch(MoviesByYearParams(released=released))


# -----------------------------------------------------------------------
# Verify the boundary: DTO has no graph internals.
# -----------------------------------------------------------------------
print(
    "MovieCard is a NodeModel:",
    issubclass(
        MovieCard,
        __import__(
            "orthograph.graph_definition.models", fromlist=["NodeModel"]
        ).NodeModel,
    ),
)
print("MovieCard fields         :", list(MovieCard.model_fields.keys()))
print("Movie    fields          :", list(Movie.model_fields.keys()))
print("__label__ on MovieCard   :", hasattr(MovieCard, "__label__"))
print("__label__ on Movie       :", hasattr(Movie, "__label__"))

# Run the route end-to-end.
r_cards = client.get("/movie-cards", params={"released": 1999})
print()
print("Status:", r_cards.status_code)
print("Body  :", r_cards.json())

assert r_cards.status_code == 200
cards = r_cards.json()
assert len(cards) == 2
for c in cards:
    assert "year" in c, "DTO should expose 'year'"
    assert "released" not in c, "graph column name must not leak into the DTO"
    assert "tagline" not in c, "NodeModel field not in DTO must not appear"
    assert "__label__" not in c

parsed_cards = [MovieCard.model_validate(c) for c in cards]
print()
print("Parsed MovieCard objects (consumer sees no graph internals):")
for c in parsed_cards:
    print(f"  {c.title} — year={c.year}")

**What changed and why it matters:**

| | `MoviesByYear` (§6) | `MovieCardsByYear` (above) |
|-|--------------------|----------------------------|
| `Output` type | `Movie` — a `NodeModel` | `MovieCard` — a plain `BaseModel` |
| Graph internals visible to consumer | Yes (`__label__`, `__uid_field__`, …) | No |
| Field rename independence | No — `released` is both stored and API name | Yes — `released` → `year` absorbed in `materialize` |
| Response schema includes `tagline` | Yes (inherited from `Movie`) | No (DTO declares only what the caller needs) |
| Route wiring | Identical `ReadPort` + `Depends()` pattern | Identical — only the generic parameters differ |

The `GraphDefinition` (containing `Movie`) is still used by `validate_cypher` to
check that the query template references valid labels and properties. That is its
only role here — it does not appear in the consumer's type signature at all.

The route below follows the same `ReadPort` + `Depends()` idiom as §6 and §9.
The three read routes in this notebook are now structurally identical at the wiring
level — only the generic parameters and the `materialize()` body change.

### Projection DTO — field subset with rename (`MovieSummary`)

The `MovieSummary` below shows the same pattern applied to a read query that
projects only a subset of fields, to contrast with the full `MovieCard` above.

In [ ]:
# --- Projection model: exposes only title + released, renames released -> year ---


class MovieSummary(BaseModel):
    """API projection of a Movie node — only the fields the caller needs."""

    title: str
    year: int  # renamed from 'released' in the graph


# --- Read query whose Output is the projection, not the NodeModel ---


class MovieSummariesByYear(CypherReadQuery[MoviesByYearParams, MovieSummary]):
    """Return movie summaries (title + year) for a given release year."""

    name = "movie_summaries_by_year"
    cypher_template = (
        "MATCH (m:Movie {released: $released}) "
        "RETURN m.title AS title, m.released AS released"
    )

    def materialize(self, raw: dict[str, Any]) -> MovieSummary:
        # Columns do NOT map 1:1 to MovieSummary (released -> year), so the
        # §4 shortcut is invalid; the mapping MUST be explicit. This is the DTO seam.
        return MovieSummary(title=raw["title"], year=raw["released"])


# Same ReadPort + Depends() pattern — structurally identical to §6 and §9.
# Only the generic parameters differ: P=MoviesByYearParams, D=MovieSummary.
def get_summaries_port() -> ReadPort[MoviesByYearParams, MovieSummary]:
    """Dependency factory for the projection read. Swap _executor here for a real driver."""
    return QueryBackedReadPort(MovieSummariesByYear(), _executor)


# --- Route using the projection ---


@app.get("/movie-summaries", response_model=list[MovieSummary])
def get_movie_summaries(
    released: int,
    port: ReadPort[MoviesByYearParams, MovieSummary] = Depends(get_summaries_port),
) -> list[MovieSummary]:
    """Return light-weight movie summaries (no tagline, renamed year field)."""
    return port.fetch(MoviesByYearParams(released=released))


r = client.get("/movie-summaries", params={"released": 1999})
print("Status:", r.status_code)
print("Body  :", r.json())

assert r.status_code == 200
summaries = r.json()
assert len(summaries) == 2

# The response has 'year', not 'released', and no 'tagline'.
for s in summaries:
    assert "year" in s
    assert "released" not in s
    assert "tagline" not in s

parsed_summaries = [MovieSummary.model_validate(s) for s in summaries]
print("\nProjected summaries:")
for s in parsed_summaries:
    print(f"  {s.title} — year={s.year}")

# Projection schema: only 'title' and 'year' — 'tagline' and 'released' absent.
proj_schema_props = set(MovieSummary.model_json_schema()["properties"].keys())
assert proj_schema_props == {"title", "year"}
print("\nProjection schema properties:", sorted(proj_schema_props))
print(
    "Full node schema properties  :",
    sorted(Movie.model_json_schema()["properties"].keys()),
)

## 10. OpenAPI schema inspection

FastAPI generates an OpenAPI schema from the route definitions. We can inspect it
in-process to confirm the response schemas are what orthograph's output models declare.

In [ ]:
openapi = app.openapi()
paths = openapi.get("paths", {})

print("Paths in OpenAPI schema:")
for path, methods in paths.items():
    for method, spec in methods.items():
        print(f"  {method.upper()} {path}")
        response_200 = spec.get("responses", {}).get("200", {})
        content = response_200.get("content", {})
        schema_ref = content.get("application/json", {}).get("schema", {}).get(
            "$ref"
        ) or content.get("application/json", {}).get("schema", {}).get("title")
        print(f"    response schema: {schema_ref}")

# Confirm the Movie and projection schemas are present in components.
components = openapi.get("components", {}).get("schemas", {})
assert "Movie" in components, "Movie schema missing from OpenAPI components"
assert "CreateMovieResult" in components, "CreateMovieResult schema missing"
assert "MovieSummary" in components, "MovieSummary schema missing"

print(
    "\nMovie schema properties        :", list(components["Movie"]["properties"].keys())
)
print(
    "CreateMovieResult props        :",
    list(components["CreateMovieResult"]["properties"].keys()),
)
print(
    "MovieSummary schema properties :",
    list(components["MovieSummary"]["properties"].keys()),
)

## 11. Limitations and future work

### What is demonstrated here

- `NodeModel` as `response_model` — no DTO needed **only when the RETURN columns
  map 1:1 to the node fields**. `__label__`, `__uid_field__` are `ClassVar` and never
  leak into JSON (§3). A separate DTO (`BaseModel` projection) is required the moment
  the API shape diverges from the node shape — the library makes DTOs opt-in for the
  1:1 case, not obsolete (§4, §9).
- `Output` class used directly as `response_model=` — no schema duplication.
- `materialize()` is the mapping seam: `Output.model_validate(raw)` for the 1:1 case,
  an explicit constructor when columns diverge (§4).
- `ReadPort` + `QueryBackedReadPort` wired via `Depends()` — used consistently
  across all three read routes (§6, §9, §9). Routes never import the executor;
  swap at the composition root (the port factory).
- Projection `BaseModel` as `Output` — the escape hatch when the API contract must
  diverge from the stored node shape (§9), using the same port pattern.
- `PaginatedParams` composes cleanly into a route signature.
- Definition-time **syntactic** Cypher validation (§2): the template parses and its
  `$param` placeholders align 1:1 with `Params`.
- `TestClient` provides a full request/response cycle in-process.

### Caveats and known gaps

- **`QueryCatalogue` is an introspection registry, not wiring infrastructure.** It can
  enumerate registered queries and expose each `Output` via `describe()` /
  `output_class`, but it does **not** compare the query contract to the route
  signature and does **not** catch contract/HTTP drift. This notebook therefore wires
  the `Output` classes directly rather than routing them through the catalogue, to
  avoid implying a guarantee the catalogue does not provide.

- **The fake executor does not run the Cypher.** It filters an in-memory list, so the
  notebook proves wiring, not query execution. The `cypher_template` strings receive
  only the §2 syntactic check here.

- **Syntactic validation will not catch a backtick-wrapped parameter** (tech-debt
  E20/T7). `` `$released` `` parses (graphglot lexes it as one escaped-identifier
  token) and passes `$param` alignment, yet binds nothing at runtime. Always write
  bare `$name`. A definition-time lint for this is tracked in E20/T7.

- **No `WritePort` / `QueryBackedWritePort`.** Write routes call `executor.write()`
  directly. This breaks the read/write symmetry: reads are injected via `Depends()`;
  writes hardcode the executor in the route body. A symmetric `WritePort` injection
  seam is planned to bring write-side wiring in line with reads.

- **Write queries cannot return the created resource.** `CypherExecutor.write()`
  intentionally discards any `RETURN` rows from the Cypher template — only mutation
  counters (`nodes_created`, `nodes_deleted`, …) are available in `interpret_result`.
  This cannot be generalised: a POST cannot return the created entity's id or full
  shape without a follow-up read query (§7). This is the most significant ergonomic
  gap for typical REST APIs and will be addressed in a future release.

- **No pagination response envelope.** There is no `PaginatedResponse[T]` wrapper
  (total count + items). Declare it in the application layer.

- **`BulkWriteQuery` is deferred.** `WriteQuery` executes a single Cypher statement per
  call. The v0.1.0 convention for bulk writes: declare `items: list[dict]` on `Params`
  and use `UNWIND $items AS item` in the template. A `BulkWriteQuery` base class is
  planned for a future release.

- **No `QueryRouter` or framework helper.** The boundary is deliberate (ADR-024):
  orthograph produces typed query contracts; consuming applications own their web
  framework wiring.

- **Live integration test deferred.** A notebook with a real Neo4j/Memgraph driver
  and a live FastAPI `TestClient` is planned once real consumer feedback is gathered
  post-public. That is the notebook that would actually exercise the `cypher_template`
  strings end-to-end.

### Import paths

```python
from orthograph.cypher.base_models import CypherReadQuery, CypherWriteQuery
from orthograph.cypher.bindings import NoParams, NoIdentifiers
from orthograph.cypher.parser import parse_cypher
from orthograph.query.base_models import ReadPort, QueryBackedReadPort
from orthograph.query.pagination import PaginatedParams
from orthograph.query.write_result import WriteResultSummary
```